In [ ]:
#Q.11 카테고리 확대·유지·축소 매트릭스
# 순매출: 취소 제외 주문의 quantity*unit_price*(1-discount) 합 (오염된 products.price 대신 실거래 unit_price 사용)
# 마진율: (순매출-원가)/순매출, 원가 = quantity*products.cost (cost는 결측 없음, 문제 13에서 확인)
# 반품률: 반품 주문수/취소 제외 주문수 (order_item 행 단위로 카테고리 귀속 - 주문이 여러 카테고리에 걸칠 수 있어서)
# -> 세 지표를 rank()로 방향 통일해 가중 종합점수 산출, 확대/유지/축소 부여
# -> 매출 단독 판단과 달라진 카테고리 지목 + 가중치 민감도 확인

In [ ]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv", usecols=["order_id", "status"], dtype={"status": "category"})
items = pd.read_csv("../data/order_items.csv")
products = pd.read_csv("../data/products.csv", usecols=["product_id", "category", "cost"])
products["category"] = products["category"].str.strip()  # " 식품" 등 앞뒤 공백 오염 정리

df = items.merge(orders, on="order_id", how="left").merge(products, on="product_id", how="left")

before = len(df)
df = df.dropna(subset=["unit_price"])
df = df[df["quantity"] > 0]  # unit_price 결측(3.0%), quantity<=0(0/음수, 데이터 오류) 라인 제외
print(f"제외된 라인: {before - len(df):,} / 전체 {before:,}")

valid = df[df["status"] != "canceled"].copy()  # 취소 주문은 매출·마진·반품률 모두에서 제외
valid["revenue"] = valid["quantity"] * valid["unit_price"] * (1 - valid["discount"])
valid["cost_amt"] = valid["quantity"] * valid["cost"]
print(f"유효(취소 제외) 주문 라인: {len(valid):,} / {len(df):,}")

In [ ]:
# 1. 카테고리별 순매출·마진율·반품률 결합
rev_margin = valid.groupby("category").agg(net_revenue=("revenue", "sum"), total_cost=("cost_amt", "sum"))
rev_margin["margin_rate"] = (rev_margin["net_revenue"] - rev_margin["total_cost"]) / rev_margin["net_revenue"]

ret = valid.groupby("category").agg(
    valid_lines=("status", "size"),
    returned=("status", lambda s: (s == "returned").sum()),
)
ret["return_rate"] = ret["returned"] / ret["valid_lines"]

cat_tbl = rev_margin[["net_revenue", "margin_rate"]].join(ret["return_rate"])
cat_tbl.sort_values("net_revenue", ascending=False).round(4)

In [ ]:
# 2. 지표별 rank()로 방향 통일(매출·마진은 클수록, 반품률은 작을수록 좋음) 후 가중 종합점수
def score_table(tbl, w_rev, w_margin, w_return):
    r_rev = tbl["net_revenue"].rank()
    r_margin = tbl["margin_rate"].rank()
    r_return = tbl["return_rate"].rank(ascending=False)
    score = w_rev * r_rev + w_margin * r_margin + w_return * r_return
    n = len(tbl)
    rk = score.rank(ascending=False, method="first")
    decision = rk.apply(lambda x: "확대" if x <= n / 3 else ("유지" if x <= 2 * n / 3 else "축소"))
    return tbl.assign(score=score, decision=decision).sort_values("score", ascending=False)

# 3. 종합 점수로 확대/유지/축소 부여 (재무 리스크인 마진·반품에 매출보다 약간 더 무게)
W_REV, W_MARGIN, W_RETURN = 0.3, 0.4, 0.3
result = score_table(cat_tbl, W_REV, W_MARGIN, W_RETURN)
result.round(4)

In [ ]:
# 매출 단독 판단과 비교 -> 결론이 달라진 카테고리 지목
n = len(cat_tbl)
rev_only = cat_tbl["net_revenue"].rank(ascending=False, method="first").apply(
    lambda x: "확대" if x <= n / 3 else ("유지" if x <= 2 * n / 3 else "축소")
)
compare = pd.DataFrame({"매출단독": rev_only, "종합점수": result["decision"]})
print("판단이 달라진 카테고리:")
compare[compare["매출단독"] != compare["종합점수"]]

In [ ]:
# 4. 가중치 민감도: 매출/마진 비중을 바꿔도 결론이 유지되는지 확인
weight_sets = {
    "기본(.3/.4/.3)": (0.3, 0.4, 0.3),
    "매출중시(.6/.2/.2)": (0.6, 0.2, 0.2),
    "마진중시(.2/.6/.2)": (0.2, 0.6, 0.2),
}
sensitivity = pd.DataFrame({name: score_table(cat_tbl, *w)["decision"] for name, w in weight_sets.items()})
sensitivity

### 제출물

- **종합표**: `result` (순매출·마진율·반품률·종합점수·결론, 위 셀)
- **제안**: 확대 = 전자·도서 / 유지 = 식품·의류 / 축소 = 가구·뷰티
- **판단이 바뀐 사례**: 가구는 매출 2위지만 마진율이 6개 카테고리 중 최저(21.6%)이고 반품률도 상위권이라 축소로 내려감. 도서는 매출 최하위권이지만 마진율 최고(29.4%)·반품률 최저라 확대로 올라감. 식품은 매출 꼴찌지만 반품률이 가장 낮아 유지로 올라감.
- **민감도**: 가중치를 매출중시/마진중시로 바꿔도 가구는 한 번도 확대로 오르지 않고 도서는 항상 확대 → 두 결론은 가중치에 안정적. 전자·식품은 가중치에 따라 등급이 갈려 합의가 더 필요한 경계 카테고리.